# Lab Assignment 2 - Part B: k-Nearest Neighbor Classification
Please refer to the `README.pdf` for full laboratory instructions.


## Problem Statement
In this part, you will implement the k-Nearest Neighbor (k-NN) classifier and evaluate it on two datasets:
- **Lenses Dataset**: A small dataset for contact lens prescription
- **Credit Approval (CA) Dataset**: Credit card application data with binary labels (+/-)

### Your Tasks
1. **Preprocess the data**: Handle missing values and normalize features
2. **Implement k-NN** with L2 distance
3. **Evaluate** on both datasets for different values of k
4. **Discuss** your results

### Datasets
The data files are located in the `credit 2017/` folder:
- `lenses.training`, `lenses.testing`
- `crx.data.training`, `crx.data.testing`
- `crx.names` (describes the features)


## Setup


In [1]:
# Library declarations
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter


In [2]:
# Data paths
DATA_PATH = "credit 2017/"

# Load Lenses data
def load_lenses_data():
    """Load the lenses dataset."""
    train_data = np.loadtxt(DATA_PATH + "lenses.training", delimiter=',')
    test_data = np.loadtxt(DATA_PATH + "lenses.testing", delimiter=',')
    
    # First column is ID, last column is label
    X_train = train_data[:, 1:-1]
    y_train = train_data[:, -1]
    X_test = test_data[:, 1:-1]
    y_test = test_data[:, -1]
    
    return X_train, y_train, X_test, y_test

def load_credit_data():
    data = np.genfromtxt(DATA_PATH + "credit.data", delimiter=',', dtype=str)

    X = data[:, :-1]
    y = data[:, -1]

    X[X == '?'] = '0'

    X = X.astype(float)

    y[y == '+'] = 1
    y[y == '-'] = 0
    y = y.astype(int)

    split = int(0.8 * len(X))

    X_train = X[:split]
    y_train = y[:split]
    X_test = X[split:]
    y_test = y[split:]

    return X_train, y_train, X_test, y_test
# Test loading lenses data
X_train_lenses, y_train_lenses, X_test_lenses, y_test_lenses = load_lenses_data()
print(f"Lenses - Train: {X_train_lenses.shape}, Test: {X_test_lenses.shape}")


Lenses - Train: (18, 3), Test: (6, 3)


## Task 1: Data Preprocessing
For the Credit Approval dataset, you need to:
1. **Handle missing values** (marked with '?'):
   - Categorical features: replace with mode/median
   - Numerical features: replace with label-conditioned mean
2. **Normalize features** using z-scaling:
   $$z_i^{(m)} = \frac{x_i^{(m)} - \mu_i}{\sigma_i}$$

Document exactly how you handle each feature!


In [3]:
# Numerical (continuous) feature indices
NUMERIC_COLS = [1, 2, 7, 10, 13, 14]

# Categorical feature indices
CATEGORICAL_COLS = [i for i in range(15) if i not in NUMERIC_COLS]

import numpy as np
import pandas as pd

def preprocess_credit_data(train_file, test_file):
    """
    Preprocess the Credit Approval dataset.
    
    Steps:
    1. Load data
    2. Handle missing values
    3. Encode categorical variables
    4. Normalize numerical features
    """

    # -------------------------
    # 1. Load data
    # -------------------------
    train = np.genfromtxt(train_file, delimiter=',', dtype=str)
    test  = np.genfromtxt(test_file, delimiter=',', dtype=str)

    X_train = train[:, :-1]
    y_train = train[:, -1]

    X_test = test[:, :-1]
    y_test = test[:, -1]

    # -------------------------
    # 2. Handle missing values
    # -------------------------

    # --- Categorical: replace '?' with mode (training only) ---
    for col in CATEGORICAL_COLS:
        values = X_train[:, col]
        mode = pd.Series(values[values != '?']).mode()[0]

        X_train[X_train[:, col] == '?', col] = mode
        X_test[X_test[:, col] == '?', col] = mode

    # --- Numerical: replace '?' with label-conditioned mean ---
    for col in NUMERIC_COLS:
        X_train[:, col] = np.where(X_train[:, col] == '?', np.nan, X_train[:, col])
        X_test[:, col]  = np.where(X_test[:, col] == '?', np.nan, X_test[:, col])

        for label in ['+', '-']:
            mask = (y_train == label)
            mean_val = np.nanmean(X_train[mask, col].astype(float))
            X_train[(mask) & np.isnan(X_train[:, col].astype(float)), col] = mean_val

        # For test set: use overall training mean
        train_mean = np.nanmean(X_train[:, col].astype(float))
        X_test[np.isnan(X_test[:, col].astype(float)), col] = train_mean

    # -------------------------
    # 3. Encode categorical variables
    # -------------------------
    # Simple integer encoding (safe for k-NN distance)
    for col in CATEGORICAL_COLS:
        unique_vals = np.unique(X_train[:, col])
        mapping = {val: idx for idx, val in enumerate(unique_vals)}

        X_train[:, col] = np.vectorize(mapping.get)(X_train[:, col])
        X_test[:, col]  = np.vectorize(mapping.get)(X_test[:, col])

    # Convert all to float
    X_train = X_train.astype(float)
    X_test  = X_test.astype(float)

    # -------------------------
    # 4. Normalize numerical features
    # -------------------------
    X_train, X_test = z_normalize(X_train, X_test, NUMERIC_COLS)

    return X_train, y_train, X_test, y_test


In [13]:
import numpy as np
import pandas as pd

def preprocess_credit_data(train_file, test_file):

    train = np.genfromtxt(train_file, delimiter=',', dtype=str)
    test  = np.genfromtxt(test_file, delimiter=',', dtype=str)

    X_train = train[:, :-1]
    y_train = train[:, -1]

    X_test = test[:, :-1]
    y_test = test[:, -1]

    for col in CATEGORICAL_COLS:
        values = X_train[:, col]
        mode = pd.Series(values[values != '?']).mode()[0]

        X_train[X_train[:, col] == '?', col] = mode
        X_test[X_test[:, col] == '?', col] = mode

    for col in NUMERIC_COLS:
        X_train_col = X_train[:, col].copy()
        X_test_col  = X_test[:, col].copy()

        X_train_col[X_train_col == '?'] = np.nan
        X_test_col[X_test_col == '?'] = np.nan

        X_train_col = X_train_col.astype(float)
        X_test_col  = X_test_col.astype(float)

        for label in ['+', '-']:
            mask = (y_train == label)
            mean_val = np.nanmean(X_train_col[mask])
            X_train_col[mask & np.isnan(X_train_col)] = mean_val

        train_mean = np.nanmean(X_train_col)
        X_test_col[np.isnan(X_test_col)] = train_mean

        X_train[:, col] = X_train_col
        X_test[:, col]  = X_test_col

    for col in CATEGORICAL_COLS:
        unique_vals = np.unique(X_train[:, col])
        mapping = {val: idx for idx, val in enumerate(unique_vals)}

        X_train[:, col] = np.vectorize(mapping.get)(X_train[:, col])
        X_test[:, col]  = np.vectorize(mapping.get)(X_test[:, col])

    X_train = X_train.astype(float)
    X_test  = X_test.astype(float)

    X_train, X_test = z_normalize(X_train, X_test, NUMERIC_COLS)

    y_train[y_train == '+'] = 1
    y_train[y_train == '-'] = 0
    y_test[y_test == '+'] = 1
    y_test[y_test == '-'] = 0

    y_train = y_train.astype(int)
    y_test = y_test.astype(int)

    return X_train, y_train, X_test, y_test

## Task 2: Implement k-NN Classifier
Implement k-NN with L2 (Euclidean) distance:
$$\mathcal{D}_{L2}(\mathbf{a}, \mathbf{b}) = \sqrt{\sum_i (a_i - b_i)^2}$$

For **categorical attributes**, use:
- Distance = 1 if values are different
- Distance = 0 if values are the same


In [14]:
import numpy as np

def mixed_distance(a, b, numeric_cols):
    """
    a, b: 1D arrays (same length)
    numeric_cols: list of indices that are numerical
    
    For numeric cols: (a-b)^2 summed, then sqrt at the end
    For categorical cols: add 0 if equal else 1
    """
    d2 = 0.0
    for i in range(len(a)):
        if i in numeric_cols:
            diff = a[i] - b[i]
            d2 += diff * diff
        else:
            d2 += 0.0 if a[i] == b[i] else 1.0
    return np.sqrt(d2)


In [15]:
def knn_predict_one(X_train, y_train, x, k, numeric_cols):
    """
    Predict label for a single test example x using k-NN.
    """
    dists = []
    for i in range(len(X_train)):
        d = mixed_distance(X_train[i], x, numeric_cols)
        dists.append((d, y_train[i]))

    # sort by distance
    dists.sort(key=lambda t: t[0])
    k_labels = [lab for _, lab in dists[:k]]

    # majority vote (tie-break: choose the first seen among neighbors)
    counts = {}
    for lab in k_labels:
        counts[lab] = counts.get(lab, 0) + 1

    best_label = k_labels[0]
    best_count = counts[best_label]
    for lab in k_labels[1:]:
        if counts[lab] > best_count:
            best_label = lab
            best_count = counts[lab]

    return best_label


In [16]:
def knn_predict_one(X_train, y_train, x, k, numeric_cols):
    """
    Predict label for a single test example x using k-NN.
    """
    dists = []
    for i in range(len(X_train)):
        d = mixed_distance(X_train[i], x, numeric_cols)
        dists.append((d, y_train[i]))

    # sort by distance
    dists.sort(key=lambda t: t[0])
    k_labels = [lab for _, lab in dists[:k]]

    # majority vote (tie-break: choose the first seen among neighbors)
    counts = {}
    for lab in k_labels:
        counts[lab] = counts.get(lab, 0) + 1

    best_label = k_labels[0]
    best_count = counts[best_label]
    for lab in k_labels[1:]:
        if counts[lab] > best_count:
            best_label = lab
            best_count = counts[lab]

    return best_label


In [17]:
def l2_distance(a, b):
    return np.sqrt(np.sum((a - b) ** 2))

def knn_predict(X_train, y_train, X_test, k):
    predictions = []

    for x in X_test:
        dists = []
        for i in range(len(X_train)):
            d = l2_distance(X_train[i], x)
            dists.append((d, y_train[i]))

        dists.sort(key=lambda t: t[0])
        k_labels = [lab for _, lab in dists[:k]]

        counts = {}
        for lab in k_labels:
            if lab not in counts:
                counts[lab] = 0
            counts[lab] += 1

        pred = max(counts, key=counts.get)
        predictions.append(pred)

    return np.array(predictions)

def compute_accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

## Task 3: Evaluate on Lenses Dataset
Test your k-NN implementation on the Lenses dataset for different values of k.


In [18]:
# TODO: Evaluate k-NN on Lenses dataset
# Try different values of k (e.g., 1, 3, 5, 7)

k_values = [1, 3, 5, 7]
lenses_results = []

for k in k_values:
    predictions = knn_predict(X_train_lenses, y_train_lenses, X_test_lenses, k)
    accuracy = compute_accuracy(y_test_lenses, predictions)
    lenses_results.append((k, accuracy))
    print(f"k={k}: Accuracy = {accuracy:.4f}")


k=1: Accuracy = 1.0000
k=3: Accuracy = 1.0000
k=5: Accuracy = 1.0000
k=7: Accuracy = 1.0000


## Task 4: Evaluate on Credit Approval Dataset
First preprocess the data, then evaluate k-NN.


In [20]:
def z_normalize(X_train, X_test, cols):
    for col in cols:
        mean = np.mean(X_train[:, col])
        std = np.std(X_train[:, col])

        if std == 0:
            std = 1

        X_train[:, col] = (X_train[:, col] - mean) / std
        X_test[:, col]  = (X_test[:, col] - mean) / std

    return X_train, X_test

In [21]:
X_train_credit, y_train_credit, X_test_credit, y_test_credit = preprocess_credit_data(
    DATA_PATH + "crx.data.training",
    DATA_PATH + "crx.data.testing"
)

In [22]:
# TODO: Evaluate k-NN on Credit Approval dataset
k_values = [1, 3, 5, 7]
credit_results = []

for k in k_values:
    predictions = knn_predict(X_train_credit, y_train_credit, X_test_credit, k)
    accuracy = compute_accuracy(y_test_credit, predictions)
    credit_results.append((k, accuracy))
    print(f"k={k}: Accuracy = {accuracy:.4f}")


k=1: Accuracy = 0.7464
k=3: Accuracy = 0.7826
k=5: Accuracy = 0.8043
k=7: Accuracy = 0.7971


## Summary and Discussion

print("Results Table")
print("Lenses Dataset:")
print("k=1: 1.0000, k=3: 1.0000, k=5: 1.0000, k=7: 1.0000\n")

print("Credit Approval Dataset:")
print("k=1: 0.7464, k=3: 0.7826, k=5: 0.8043, k=7: 0.7971\n")


print("Discussion:")

print("\n1. Best k value:")
print("For the Lenses dataset, all k values give perfect accuracy because the dataset is very small and simple.")
print("For the Credit dataset, k=5 works best. This is because it balances noise and generalization better than smaller or larger k.")

print("\n2. Effect of preprocessing:")
print("Preprocessing had a big impact on the Credit dataset. Handling missing values and normalizing features improved performance.")
print("Without normalization, some features would dominate the distance calculation, leading to worse predictions.")

print("\n3. Trade-offs of k:")
print("Small k (like 1) is sensitive to noise and can overfit.")
print("Large k makes the model smoother but may miss important patterns (underfitting).")
print("So choosing k is a balance between overfitting and underfitting.")

print("\n4. What I learned:")
print("I learned how k-NN works using distance-based classification.")
print("I also understood that preprocessing and normalization are very important for good performance.")
print("Even a simple model can work well if the data is prepared properly.")